In [22]:
from os import chdir
from pathlib import Path

cwd = Path.cwd()
print(f"CWD: {cwd}")
if cwd.name == "Sota"or cwd.name == "ML100K":
    chdir("/Users/haowen/Documents/Decentral RS/fed-learning-main")
print(f"CWD: {Path.cwd()}")

CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main
CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main


In [24]:
import pandas as pd
import torch
from sklearn.utils import shuffle

In [26]:
train = pd.read_csv("dataset/ml100k_train.csv")
test = pd.read_csv("dataset/ml100k_test.csv")
train.head()

,user_id,item_id,rating,rating.1
0,207,87,5,5
1,675,900,4,4
2,373,230,2,2
3,377,564,3,3
4,725,843,3,3


In [28]:
rating_matrix = train.pivot(index='user_id', columns='item_id', values='rating')
n_users, n_movies = rating_matrix.shape
# Scaling ratings to between 0 and 1, this helps our model by constraining predictions
min_rating, max_rating = train['rating'].min(), train['rating'].max()
rating_matrix = (rating_matrix - min_rating) / (max_rating - min_rating)
print(rating_matrix)
print(n_users*n_movies-rating_matrix.isnull().values.sum())

item_id  0     1     2     3     4     5     6     7     8     9     ...  \
user_id                                                              ...   
0        1.00   0.5  0.75  0.50   NaN   1.0   NaN   0.0   NaN  0.50  ...   
1         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  0.25  ...   
2         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
3         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
4        0.75   0.5   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
...       ...   ...   ...   ...   ...   ...   ...   ...   ...   ...  ...   
938       NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   1.0   NaN  ...   
939       NaN   NaN   NaN  0.25   NaN   NaN  0.75   1.0   NaN   NaN  ...   
940       NaN   NaN   NaN   NaN   NaN   NaN  0.75   NaN   NaN   NaN  ...   
941       NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
942       NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   

item_id  16

In [30]:
# Replacing missing ratings with -1 so we can filter them out later

rating_matrix[rating_matrix.isnull()] = -1
rating_matrix = torch.FloatTensor(rating_matrix.values)
print(rating_matrix)

tensor([[ 1.0000,  0.5000,  0.7500,  ..., -1.0000, -1.0000, -1.0000],
        [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
        [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
        ...,
        [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
        [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
        [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000]])


In [32]:
# # This is how we can define our feature matrices
# # We're going to be training these, so we'll need gradients
# latent_vectors = 5
# user_features = torch.randn(n_users, latent_vectors, requires_grad=True)
# user_features.data.mul_(0.01)
# movie_features = torch.randn(n_movies, latent_vectors, requires_grad=True)
# movie_features.data.mul_(0.01)

In [34]:
class FCFLoss(torch.nn.Module):
    def __init__(self, lam_u=0.3, lam_v=0.3):
        super().__init__()
        self.lam_u = lam_u
        self.lam_v = lam_v
    
    def forward(self, matrix, u_features, v_features):
        non_zero_mask = (matrix != -1).type(torch.FloatTensor)
        predicted = torch.sigmoid(torch.mm(u_features, v_features.t()))
        
        diff = (matrix - predicted)**2
        prediction_error = torch.sum(diff*non_zero_mask)

        u_regularization = self.lam_u * torch.sum(u_features.norm(dim=1))
        v_regularization = self.lam_v * torch.sum(v_features.norm(dim=1))
        
        return prediction_error + u_regularization + v_regularization, prediction_error

In [36]:
# FCF

num_client = 200
m = n_users//num_client
num_epoch = 100   
latent_vectors = 20
# user_features = torch.randn(n_users, latent_vectors, requires_grad=True)
# user_features.data.mul_(0.01)

user_features = []
for i in range(num_client):
    user_features.append(torch.randn(m, latent_vectors, requires_grad=True))
# user_features = [torch.randn(m, latent_vectors, requires_grad=True)]*num_client
movie_features = torch.randn(n_movies, latent_vectors, requires_grad=True)
movie_features.data.mul_(0.01)

fcferror = FCFLoss(lam_u=0.1, lam_v=0.1)
# optimizer_local = torch.optim.Adam([user_features], lr=0.01)

optimizer_client = []

# print(m)
for i in range(num_client):
    optimizer = torch.optim.Adam([user_features[i]], lr=0.01)
    optimizer_client.append(optimizer)

error_list = []
previous = torch.zeros(n_movies, latent_vectors)
optimizer_server = torch.optim.Adam([movie_features], lr=0.01)
# print(optimizer_set)
for step, epoch in enumerate(range(num_epoch)):
    optimizer_server.zero_grad()
    # local update
    total_loss = 0
    aver_prediction_error = 0
    for i in range(num_client):
        optimizer_client[i].zero_grad()
        loss, prediction_error = fcferror(rating_matrix[i*m:(i+1)*m], user_features[i], movie_features)
        aver_prediction_error += prediction_error/num_client
        total_loss += loss
        # print(movie_features[0])
        loss.backward()
        optimizer_client[i].step()
        
    # server update
    optimizer_server.step()

    # stop criterion
    error = (torch.norm(movie_features-previous,p=2))/torch.norm(previous,p=2)
    # print(error.detach().numpy())
    error_list.append(error.detach().numpy())
    previous = movie_features.clone()

    if step % 10 == 0:
        print(f"Step {step}, {aver_prediction_error:.3f}")
        # print(f"Step {step}, {total_loss:.3f}")

Step 0, 31.097
Step 10, 30.942
Step 20, 30.996
Step 30, 31.002
Step 40, 30.986
Step 50, 30.906
Step 60, 30.608
Step 70, 29.670
Step 80, 27.626
Step 90, 25.396


In [40]:
test_rating_matrix = test.pivot(index='user_id', columns='item_id', values='rating')

test_rating_matrix[test_rating_matrix.isnull()] = -1
test_rating_matrix = torch.FloatTensor(test_rating_matrix.values)
test_rating_matrix.shape

torch.Size([943, 1408])

In [42]:
train_items = train["item_id"].drop_duplicates().sort_values().reset_index(drop=True)
test_items = test["item_id"].drop_duplicates().sort_values().reset_index(drop=True)
# Create a mapping from item_id → index in train_items
item_to_index = pd.Series(train_items.index.values, index=train_items.values)

# Map test_items to train indices
test_indices_in_train = test_items.map(item_to_index)

print(test_indices_in_train)

0          0
1          1
2          2
3          3
4          4
        ... 
1403    1593
1404    1596
1405    1606
1406    1608
1407    1620
Name: item_id, Length: 1408, dtype: int64


In [44]:
print(movie_features)

tensor([[-8.5631e-02,  4.8124e-02,  2.8313e-02,  ..., -2.0370e-02,
         -7.6730e-02, -3.7390e-02],
        [-4.5641e-04, -2.1215e-05,  1.8952e-04,  ..., -6.1634e-05,
         -2.8524e-05,  3.9143e-04],
        [ 3.9017e-04,  4.8048e-05,  4.8538e-04,  ...,  1.1501e-04,
          1.0250e-03, -1.3403e-04],
        ...,
        [ 8.3602e-05, -1.3882e-04, -8.1255e-04,  ...,  1.1927e-04,
          1.6662e-04,  1.1361e-04],
        [-8.4497e-04,  8.8690e-05,  1.0893e-03,  ...,  1.9812e-05,
          1.9865e-04, -1.9507e-04],
        [-1.4472e-04, -6.2235e-05, -7.3134e-04,  ..., -1.5878e-04,
         -3.0023e-05,  9.4445e-05]], requires_grad=True)


In [46]:
non_zero_mask = (test_rating_matrix != -1).type(torch.FloatTensor)
num = torch.sum(non_zero_mask)
def Error(matrix, u_features, v_features):
    predicted_ratings = torch.sigmoid(torch.mm(u_features,v_features.t()))
    predicted_ratings = predicted_ratings[:,test_indices_in_train]
    # predicted_ratings = non_zero_mask * predicted_ratings
    pred = (predicted_ratings*(max_rating - min_rating) + min_rating)*non_zero_mask[i*m:(i+1)*m]
    actual = matrix*non_zero_mask[i*m:(i+1)*m]
    # print(pred)
    # print(actual)
    AE_diff = torch.abs(pred - actual)
    SE_diff = (pred - actual)**2
    
    prediction_abs_error = torch.sum(AE_diff)
    prediction_squared_error = torch.sum(SE_diff)
    return prediction_abs_error, prediction_squared_error

AE_error = 0
SE_error = 0

for i in range(num_client):
    AE_error += Error(test_rating_matrix[i*m:(i+1)*m], user_features[i], movie_features)[0]
    SE_error += Error(test_rating_matrix[i*m:(i+1)*m], user_features[i], movie_features)[1]
test_MAE = AE_error/num
test_RMSE = torch.sqrt(SE_error/num)
print('test_MAE =', test_MAE.data.numpy())
print('test_RMSE =', test_RMSE.data.numpy())

test_MAE = 0.7593413
test_RMSE = 1.0268927


In [25]:
import numpy as np
def topk_metrics(y_true, y_pred, topKs=[10]):
	"""choice topk metrics and compute it
	the metrics contains 'ndcg', 'mrr', 'recall' and 'hit'

	Args:
		y_true: list, 2-dim, each row contains the items that the user interacted
		y_pred: list, 2-dim, each row contains the items recommended  
		topKs: list or tuple, if you want to get top5 and top10, topKs=(5, 10)

	Return:
		results: list, it contains five metrics, 'ndcg', 'recall', 'mrr', 'hit', 'precision'

	"""
	assert len(y_true) == len(y_pred)

	if not isinstance(topKs, (tuple, list)):
		raise ValueError('topKs wrong, it should be tuple or list')

	ndcg_result = []
	mrr_result = []
	hit_result = []
	precision_result = []
	recall_result = []
	for idx in range(len(topKs)):
		ndcgs = 0
		mrrs = 0
		hits = 0
		precisions = 0
		recalls = 0
		for i in range(len(y_true)):
			if len(y_true[i]) != 0:
				mrr_tmp = 0
				mrr_flag = True
				hit_tmp = 0
				dcg_tmp = 0
				idcg_tmp = 0
				hit = 0
				for j in range(topKs[idx]):
					if y_pred[i][j] in y_true[i]:
						hit += 1.
						if mrr_flag:
							mrr_flag = False
							mrr_tmp = 1. / (1 + j)
							hit_tmp = 1.
						dcg_tmp += 1. / (np.log2(j + 2))
					idcg_tmp += 1. / (np.log2(j + 2))
				hits += hit_tmp
				mrrs += mrr_tmp
				recalls += hit / len(y_true[i])
				precisions += hit / topKs[idx]
				if idcg_tmp != 0:
					ndcgs += dcg_tmp / idcg_tmp
		hit_result.append(round(hits / len(y_pred), 4))
		mrr_result.append(round(mrrs / len(y_pred), 4))
		recall_result.append(round(recalls / len(y_pred), 4))
		precision_result.append(round(precisions / len(y_pred), 4))
		ndcg_result.append(round(ndcgs / len(y_pred), 4))

	results = []
	for idx in range(len(topKs)):
		results.append(hit_result[idx])
	return results

In [28]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

non_zero_mask = (test_rating_matrix != -1).type(torch.FloatTensor)
num = torch.sum(non_zero_mask)
print(num)
def recommend(u_features, v_features, mask, k):
    scores = (torch.sigmoid(torch.mm(u_features,v_features.t()))).data.numpy()
    # scores = (torch.sigmoid(torch.mm(u_features,v_features.t()))).data.numpy()
    recommend_list = []
    # print(scores)
    for i in range(len(scores)):
    # print(torch.sum(mask))
    # print(scores.shape)
        score = scores[i]
        sorted_id = sorted(range(len(score)), key=lambda k: score[k], reverse=True)[:k]
        recommend_list.append(sorted_id)
    return recommend_list
def ground_truth(actual):
    actual_list = []
    for i in range(len(actual)):
        actual_id = [j for j, e in enumerate(actual[i]) if e != 0]
        actual_list.append(actual_id)
    return actual_list


tensor(25000.)


In [30]:
Hit_Ratio_10 = 0
for i in range(num_client):
    actual_matrix = (test_rating_matrix[i*m:(i+1)*m]*non_zero_mask[i*m:(i+1)*m]).data.numpy()
    # print(actual_matrix)
    actual = ground_truth(actual_matrix)
    predict = recommend(user_features[i], movie_features,non_zero_mask[i*m:(i+1)*m],10)
    results = topk_metrics(actual, predict,topKs=(5,10))
    # print(results)
    Hit_Ratio_10 += results[1]/num_client
    # print(i)
print(f'HR@{10}: {Hit_Ratio_10}')

HR@10: 0.5699999999999986


In [31]:
def score_f(u_features, v_features, mask, k):
    scores = (torch.sigmoid(torch.mm(u_features,v_features.t()))).data.numpy()
    # scores = (torch.sigmoid(torch.mm(u_features,v_features.t()))).data.numpy()
    recommend_list = []
    score_list = []
    # print(scores)
    for i in range(len(scores)):
    # print(torch.sum(mask))
    # print(scores.shape)
        score = scores[i]
        sorted_id = sorted(range(len(score)), key=lambda k: score[k], reverse=True)[:k]
        sorted_score = [score[j] for j in sorted_id]
        recommend_list.append(sorted_id)
        score_list.append(sorted_score)
    return recommend_list, score_list
def rel(actual_matrix, recommend):
    rel_list = []
    for i in range(len(actual_matrix)):
        actual_vector = actual_matrix[i]
        rel_id = [actual_vector[j] for j in recommend[i]]
        rel_list.append(rel_id)
    return rel_list

In [32]:
import numpy as np
from sklearn import metrics

ndcg_10 = 0
ndcg_20 = 0
for i in range(num_client):
# for i in range(1):
    actual_matrix = (test_rating_matrix[i*m:(i+1)*m]*non_zero_mask[i*m:(i+1)*m]).data.numpy()
    # print(actual_matrix)
    recommend, score = score_f(user_features[i], movie_features,non_zero_mask[i*m:(i+1)*m],10)
    true_relevance = rel(actual_matrix, recommend)

    ndcg_10 += metrics.ndcg_score(true_relevance, score, k=10)/num_client
    # ndcg_20 += metrics.ndcg_score(true_relevance, score, k=20)/num_client
print(f'NDCG@{10}: {ndcg_10}')
# print(f'NDCG@{20}: {ndcg_20}')

NDCG@10: 0.32521590761586633
